# The Naive Bayes Algorithm

This notebook turns the one-word spam detector into a real classifier that combines **many features** — and introduces the *naive assumption* that makes the math tractable. It is based on the fourteenth video of Week 1 (Introduction to Probability).


## 1. Introduction

One word is not enough to build a classifier. We want to combine "lottery" with "winning" — and eventually with hundreds of words. The direct approach would be:

$$
P(\text{spam} \mid \text{lottery} \cap \text{winning}) = \frac{\text{spam emails containing both words}}{\text{all emails containing both words}}
$$

But scale this to 100 words $w_1, \dots, w_{100}$: we would need to count emails containing **all 100 words**. Probably no email in the dataset contains all of them, giving the useless $\frac{0}{0}$. Direct counting breaks.

After completing this notebook you will be able to:

- Explain why direct counting fails for many features.
- State the naive (conditional independence) assumption and why it is "naive".
- Compute naive Bayes posteriors for any number of word features.


## 2. Theory

### The naive assumption

If we cannot *count* $P(\text{lottery} \cap \text{winning} \mid \text{spam})$, we can **estimate** it. The naive assumption says: *given the class, word appearances are independent of each other.* Then the product rule for independent events applies inside each class:

$$
P(\text{lottery} \cap \text{winning} \mid \text{spam}) \approx P(\text{lottery} \mid \text{spam}) \cdot P(\text{winning} \mid \text{spam})
$$

This is obviously **not true** — words depend on each other ("good" makes "morning" more likely, because "good morning" travels as a pair). But assuming independence makes the math work out, and the results are surprisingly good. That's why the algorithm is called **naive** Bayes.

### The formula for $n$ words

$$
P(\text{spam} \mid w_1, \dots, w_n) =
\frac{P(\text{spam}) \prod_{i=1}^{n} P(w_i \mid \text{spam})}
{P(\text{spam}) \prod_i P(w_i \mid \text{spam}) + P(\text{ham}) \prod_i P(w_i \mid \text{ham})}
$$

Each factor $P(w_i \mid \text{class})$ is a simple one-word count — easy to measure even in a small dataset.


## 3. Worked Example

The dataset from notebook 12, extended with a second word:

| Quantity | Count | Value |
|---|---|---|
| $P(\text{spam})$ | 20 of 100 | 0.2 |
| $P(\text{ham})$ | 80 of 100 | 0.8 |
| $P(\text{lottery} \mid \text{spam})$ | 14 of 20 | 0.7 |
| $P(\text{lottery} \mid \text{ham})$ | 10 of 80 | 0.125 |
| $P(\text{winning} \mid \text{spam})$ | 15 of 20 | 0.75 |
| $P(\text{winning} \mid \text{ham})$ | 8 of 80 | 0.1 |

Plug into the naive Bayes formula:

$$
P(\text{spam} \mid \text{both}) =
\frac{0.2 \times 0.7 \times 0.75}
{0.2 \times 0.7 \times 0.75 + 0.8 \times 0.125 \times 0.1}
= \frac{0.105}{0.105 + 0.010} \approx 0.913
$$

An email containing *both* words is spam with probability **91.3%** — much higher than the 58.3% from "lottery" alone. Each additional spammy word compounds the evidence.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)


### A general naive Bayes function

`words` maps each word to its pair of likelihoods $(P(w \mid \text{spam}), P(w \mid \text{ham}))$.


In [ ]:
WORDS = {
    "lottery": (14 / 20, 10 / 80),
    "winning": (15 / 20, 8 / 80),
}

def naive_bayes(present_words, p_spam=0.2, words=WORDS):
    spam_term, ham_term = p_spam, 1 - p_spam
    for w in present_words:
        p_w_spam, p_w_ham = words[w]
        spam_term *= p_w_spam
        ham_term *= p_w_ham
    return spam_term / (spam_term + ham_term)

print(f"P(spam | lottery)          = {naive_bayes(['lottery']):.3f}")
print(f"P(spam | winning)          = {naive_bayes(['winning']):.3f}")
print(f"P(spam | lottery, winning) = {naive_bayes(['lottery', 'winning']):.3f}")


### Why direct counting fails but naive Bayes doesn't

We generate a synthetic dataset of 100 emails where each word appears independently within its class (so the naive assumption holds by construction), then compare direct counting against the naive estimate as words are added.


In [ ]:
n_words = 10
p_word_spam, p_word_ham = 0.7, 0.125

n_spam, n_ham = 20, 80
spam_matrix = rng.random((n_spam, n_words)) < p_word_spam
ham_matrix = rng.random((n_ham, n_words)) < p_word_ham

for k in [1, 2, 3, 5, 10]:
    # direct counting: emails containing ALL first k words
    spam_all = np.all(spam_matrix[:, :k], axis=1).sum()
    ham_all = np.all(ham_matrix[:, :k], axis=1).sum()
    direct = f"{spam_all}/{spam_all + ham_all}" if spam_all + ham_all else "0/0 !"

    # naive Bayes estimate
    s = 0.2 * p_word_spam ** k
    h = 0.8 * p_word_ham ** k
    print(f"{k:>2} words:  direct count = {direct:<8}  naive Bayes = {s/(s+h):.4f}")


## 5. Visualizations

### Evidence compounds

Each additional spammy word multiplies the spam-vs-ham ratio; the posterior races toward 1.


In [ ]:
ks = np.arange(0, 8)
posteriors = []
for k in ks:
    s = 0.2 * 0.7 ** k
    h = 0.8 * 0.125 ** k
    posteriors.append(s / (s + h))

plt.figure(figsize=(8, 5))
plt.plot(ks, posteriors, "o-")
plt.axhline(0.5, color="gray", linestyle=":")
for k, p in zip(ks[:4], posteriors[:4]):
    plt.annotate(f"{p:.3f}", (k, p), textcoords="offset points",
                 xytext=(8, -12), fontsize=10)
plt.title("P(spam | k spammy words): 0.2 → 0.583 → 0.913 → 0.983 → ...")
plt.xlabel("Number of spammy words present (each like 'lottery')")
plt.ylabel("Posterior probability of spam")
plt.grid(True, alpha=0.3)
plt.show()


### The two words as coordinates

Placing the four kinds of emails on a grid of likelihoods shows what the classifier does: it multiplies per-word ratios. Words to the upper-left of the diagonal favor spam.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))

for word, (p_s, p_h) in WORDS.items():
    ax.scatter(p_h, p_s, s=120, zorder=3)
    ax.annotate(f"{word}\n({p_h}, {p_s})", (p_h, p_s),
                textcoords="offset points", xytext=(12, 0), fontsize=11)

ax.plot([0, 1], [0, 1], "--", color="gray", label="P(w|spam) = P(w|ham): uninformative")
ax.fill_between([0, 1], [0, 1], 1, alpha=0.08, color="red")
ax.text(0.25, 0.8, "evidence for spam", fontsize=11, color="darkred")
ax.text(0.55, 0.25, "evidence for ham", fontsize=11, color="darkblue")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("P(word | ham)")
ax.set_ylabel("P(word | spam)")
ax.set_title("Both course words sit deep in spam territory")
ax.grid(True, alpha=0.3)
plt.show()


## 6. Exercises

### Basic

1. Using the table, what is $P(\text{spam} \mid \text{winning})$ alone? (Single-word Bayes, as in notebook 12.)
2. Why is the algorithm called *naive*?

### Intermediate

3. Add a third word: "meeting" appears in 1 of 20 spam emails and 40 of 80 ham emails. What is $P(\text{spam} \mid \text{lottery, winning, meeting})$? Does one hammy word outweigh two spammy ones?
4. What happens to the naive Bayes estimate if a word appears in **0** spam emails and you multiply by $P(w \mid \text{spam}) = 0$? Suggest a fix. (This is why real implementations use *Laplace smoothing* — add 1 to every count.)

### Advanced

5. Simulate 100,000 emails from the synthetic model above (10 words, naive assumption true). Classify each with naive Bayes at threshold 0.5 and measure the accuracy. Then repeat with words made *dependent* within a class (duplicate one word's column) and observe how accuracy changes.


## 7. Solutions


In [ ]:
# Exercise 1
print(f"1.  P(spam | winning) = {naive_bayes(['winning']):.3f}")

# Exercise 2
print("2.  Because it assumes words appear independently within each class, "
      "which is false in real language.")

# Exercise 3
WORDS3 = dict(WORDS, meeting=(1 / 20, 40 / 80))
p3 = naive_bayes(["lottery", "winning", "meeting"], words=WORDS3)
print(f"3.  P(spam | all three) = {p3:.3f} — 'meeting' (20x more common in ham) "
      "drags the posterior below the two-word 0.913")

# Exercise 4
print("4.  The spam term becomes 0 no matter how spammy other words are. "
      "Fix: Laplace smoothing, e.g. P(w|spam) = (count+1)/(20+2).")

# Exercise 5: accuracy under the naive assumption vs broken assumption
def simulate(dependent):
    n = 100_000
    is_spam = rng.random(n) < 0.2
    p = np.where(is_spam[:, None], 0.7, 0.125)
    x = rng.random((n, 10)) < p
    if dependent:
        x[:, 1] = x[:, 0]        # word 2 duplicates word 1: fully dependent
    log_s = np.log(0.2) + (x * np.log(0.7) + ~x * np.log(0.3)).sum(axis=1)
    log_h = np.log(0.8) + (x * np.log(0.125) + ~x * np.log(0.875)).sum(axis=1)
    predicted_spam = log_s > log_h
    return (predicted_spam == is_spam).mean()

print(f"5.  Accuracy, assumption true: {simulate(False):.4f}; "
      f"with a duplicated word: {simulate(True):.4f} — "
      "double-counted evidence hurts only slightly.")


## 8. Conclusion

- Direct counting of multi-word intersections collapses ($\frac{0}{0}$) as features grow; **naive Bayes** replaces the joint likelihood with a product of per-word likelihoods.
- The naive assumption — conditional independence of words given the class — is false for real language but works remarkably well in practice.
- With the course's numbers, "lottery" + "winning" pushes the posterior from the 0.2 prior to **0.913**.
- Practical refinements exist for the corner cases: zero counts are handled with Laplace smoothing, and real implementations multiply in log-space to avoid underflow with hundreds of words.
